# From Python containers to NumPy arrays and pandas tables

Represent a heat-diffusion experiment as numerical arrays and labeled observations while preserving
shape, units, missingness, and alignment.

**Lecture 4 · Notebook 00 · CMOR 438 / INDE 577**

## Orientation: two libraries, two complementary mental models

**Live core:** array structure, dtype, axes, indexing, vectorization, broadcasting, `Series`,
`DataFrame`, indexes, alignment, selection, missing values, grouping, and the array/table boundary.

**Practice:** predict shapes, diagnose alignment, validate a measurement table, and write a small
summary function.

**Extension:** random-number generators, memory ownership, categorical data, and a design exercise.

NumPy and pandas are large libraries. This notebook teaches the mechanisms that prevent silent
scientific errors; later notebooks deepen indexing, reshaping, joins, and analytical workflows.

## How to use this notebook

**Estimated time:** 135 minutes core, plus 75 minutes of practice and extension.

**Prerequisites:** Python lists, dictionaries, functions, exceptions, packages, environments, and
tests. Run `uv sync` after pulling changes, select the **Rice DSM** kernel in VS Code, then restart
and run all cells.

Before each code cell, predict the object type, shape, labels, dtype, and whether missing values are
possible. After it runs, compare the result with your prediction and read every assertion as a claim.

## Learning objectives

By the end, you should be able to:

- explain why homogeneous arrays and labeled tables complement built-in containers;
- create an `ndarray` and interpret its `ndim`, `shape`, `size`, `dtype`, and axes;
- predict the shape produced by indexing, slicing, reduction, and broadcasting;
- distinguish scalar, basic, Boolean, and advanced indexing and recognize view/copy risk;
- replace element-by-element Python loops with array operations when semantics permit;
- construct and inspect `Series`, `Index`, and `DataFrame` objects;
- distinguish label-based `.loc` selection from position-based `.iloc` selection;
- predict pandas alignment by labels and detect missing or duplicated identifiers;
- make missing-value, dtype, and aggregation policy explicit;
- move deliberately between pandas and NumPy without losing labels unnoticed; and
- test shape, schema, units, alignment, and numerical invariants.

## Why this matters

Machine-learning code eventually expects numerical arrays, while scientific data usually arrives
with identifiers, units, categories, timestamps, and missing observations. NumPy is strongest when
the computation is naturally an operation on a homogeneous rectangular array. pandas is strongest
when labels and heterogeneous columns carry analytical meaning.

Many expensive defects live at their boundary: a feature matrix can have the expected shape while
its columns are reordered; two `Series` can have the same length while their entity labels differ;
and a missing sensor value can be silently converted or dropped. Representation is part of the
scientific argument.

## Worked example: a small heat-diffusion experiment

We observe temperature at three sensors placed different distances from a heat source. Two materials
are measured over four times. Our teaching data are deterministic and synthetic: an exponential
decay model supplies a smooth signal and fixed small deviations imitate measurement error.

```text
scientific question
    → how does cooling differ by material, time, and sensor distance?
mathematical representation
    → vectors, matrices, and grouped observations
software representation
    → ndarray for homogeneous computation; DataFrame for labeled records
evidence
    → shape/schema checks, missingness policy, summaries, and tests
```

This dataset demonstrates representation; it is not evidence about real copper or aluminum.

## Professional practice

| Data scientist asks | Software engineer asks |
| --- | --- |
| What does one row and one axis represent? | Where are shape and schema contracts checked? |
| What physical unit does each value use? | Can names and types preserve that unit? |
| Does missing mean absent, failed, or below detection? | Is missingness represented and handled explicitly? |
| Should observations align by order or identity? | Which labels are unique and which operation performs alignment? |
| Is an aggregate scientifically meaningful? | Which axis, groups, and null policy does the public interface (API) encode? |
| Did preprocessing preserve entity boundaries? | Can tests detect row, column, or index misalignment? |

Fast numerical code that attaches results to the wrong entities is incorrect.

## 1. Confirm the environment before studying the objects

The import names are `numpy` and `pandas`; the conventional aliases are `np` and `pd`. These aliases
are community conventions, not Python keywords. The distributions are declared in `pyproject.toml`
and resolved in `uv.lock`, so the course kernel and CI install the same versions.

In [ ]:
from math import prod

import numpy as np
import pandas as pd
import pytest

print("NumPy version:", np.__version__)
print("pandas version:", pd.__version__)

assert int(np.__version__.split(".")[0]) >= 2
assert int(pd.__version__.split(".")[0]) >= 3

If either import fails, do not install into an unknown interpreter. In VS Code, confirm the notebook
kernel is **Rice DSM**, run `uv sync` in the project root, and restart the kernel. Lecture 3 Notebook 00 explains
why package installation and kernel selection are separate facts.

## 2. Why not use only lists and dictionaries?

Built-in containers remain excellent for control flow, irregular records, and general Python
objects. But a list does not record a numerical dtype or multidimensional shape, and `+` means
concatenation rather than elementwise addition.

In [ ]:
python_temperatures = [76.4, 68.1, 59.7]

assert python_temperatures + python_temperatures == [
    76.4,
    68.1,
    59.7,
    76.4,
    68.1,
    59.7,
]

try:
    python_temperatures - 22.0
except TypeError as error:
    print(type(error).__name__, error)

NumPy gives array operations a mathematical interpretation. That convenience is earned by stronger
constraints: arrays are rectangular, have a fixed number of elements, and usually store one dtype.
Choose the representation whose constraints match the problem.

## 3. The `ndarray`: data buffer plus metadata

An `ndarray` is an N-dimensional rectangular array. Conceptually it combines homogeneous element
data with metadata describing dtype, shape, and memory layout.

```text
object: sensor_temperatures_c

axis 0 (time) ↓       axis 1 (sensor) →
                     S1      S2      S3
time 0 s           76.4    72.2    64.8
time 30 s          58.6    55.1    49.9
time 60 s          44.9    42.7    39.4
time 90 s          35.7    34.5    32.8

shape = (4, 3): four positions along axis 0, three along axis 1
```

An axis number is not inherently “row” or “feature.” Its meaning comes from our contract.

In [ ]:
sensor_temperatures_c = np.array(
    [
        [76.4, 72.2, 64.8],
        [58.6, 55.1, 49.9],
        [44.9, 42.7, 39.4],
        [35.7, 34.5, 32.8],
    ],
    dtype=np.float64,
)

print(sensor_temperatures_c)
print("ndim:", sensor_temperatures_c.ndim)
print("shape:", sensor_temperatures_c.shape)
print("size:", sensor_temperatures_c.size)
print("dtype:", sensor_temperatures_c.dtype)

assert sensor_temperatures_c.ndim == 2
assert sensor_temperatures_c.shape == (4, 3)
assert sensor_temperatures_c.size == prod(sensor_temperatures_c.shape) == 12
assert np.issubdtype(sensor_temperatures_c.dtype, np.floating)

### Shape is part of the meaning

The same 12 numbers could represent four times × three sensors, three sensors × four times, or twelve
independent observations. NumPy knows only `(4, 3)`, not the domain meaning. Use names, documentation,
and assertions to keep axis semantics visible.

**Prediction:** What are the shapes of one row, one column, and one scalar? Check before running.

In [ ]:
first_time = sensor_temperatures_c[0, :]
first_sensor = sensor_temperatures_c[:, 0]
one_temperature = sensor_temperatures_c[0, 0]

print(type(first_time), first_time.shape)
print(type(first_sensor), first_sensor.shape)
print(type(one_temperature), np.shape(one_temperature))

assert first_time.shape == (3,)
assert first_sensor.shape == (4,)
assert np.ndim(one_temperature) == 0

Indexing with an integer removes that axis. Slicing preserves it. Therefore
`array[0, :]` has shape `(3,)`, while `array[0:1, :]` has shape `(1, 3)`. This distinction matters
when another function expects a batch dimension.

In [ ]:
one_time_vector = sensor_temperatures_c[0, :]
one_time_matrix = sensor_temperatures_c[0:1, :]

assert one_time_vector.shape == (3,)
assert one_time_matrix.shape == (1, 3)

### Array construction communicates intent

- `np.array` converts explicit values.
- `np.zeros`, `np.ones`, and `np.full` initialize a known shape.
- `np.arange` advances by a step and is most predictable for integers.
- `np.linspace` requests a fixed number of evenly spaced values including endpoints by default.
- `reshape` changes the shape without changing the element count.

Avoid `np.empty` until you understand that its entries are uninitialized memory, not zeros.

In [ ]:
measurement_times_s = np.arange(0, 120, 30, dtype=np.int64)
sensor_distances_cm = np.array([1.0, 2.0, 4.0])
model_times_s = np.linspace(0.0, 90.0, num=7)
quality_weights = np.ones(sensor_temperatures_c.shape, dtype=np.float64)
missing_codes = np.full(sensor_temperatures_c.shape, fill_value=-1, dtype=np.int8)

assert np.array_equal(measurement_times_s, np.array([0, 30, 60, 90]))
assert model_times_s.shape == (7,)
assert quality_weights.shape == sensor_temperatures_c.shape
assert missing_codes.dtype == np.int8

### Dtype is a storage and numerical contract

The dtype controls representation, memory, range, and arithmetic. NumPy chooses a common dtype when
constructing an array. Mixing integers and floats normally promotes to floating point; mixing numbers
and text may produce a string dtype instead of an error.

In [ ]:
integer_array = np.array([1, 2, 3])
promoted_array = np.array([1, 2.5, 3])
text_array = np.array([1, "sensor-failed", 3])

print(integer_array.dtype, promoted_array.dtype, text_array.dtype)

assert np.issubdtype(integer_array.dtype, np.integer)
assert np.issubdtype(promoted_array.dtype, np.floating)
assert np.issubdtype(text_array.dtype, np.str_)

Never infer scientific validity from “NumPy accepted it.” Inspect dtype after ingestion. A narrow
integer dtype can overflow; floating-point values are approximate; and strings inside a supposed
feature matrix usually indicate a boundary error.

Ragged nested sequences are not rectangular arrays. Modern NumPy rejects them unless asked for
`dtype=object`, which would give up normal homogeneous numerical behavior.

In [ ]:
try:
    np.array([[1.0, 2.0], [3.0]])
except ValueError as error:
    print(type(error).__name__, error)

## 4. Selection: scalar, slice, Boolean mask, and index arrays

Basic indexing uses integers and slices. A Boolean mask selects positions where its entries are
`True`. Integer index arrays select an explicit set or ordering of positions.

Selection changes both values and shape, so inspect both.

In [ ]:
late_time_rows = sensor_temperatures_c[2:, :]
above_50_c = sensor_temperatures_c[sensor_temperatures_c > 50.0]
reordered_sensors = sensor_temperatures_c[:, [2, 0]]

assert late_time_rows.shape == (2, 3)
assert above_50_c.ndim == 1
assert np.all(above_50_c > 50.0)
assert reordered_sensors.shape == (4, 2)
assert np.array_equal(reordered_sensors[:, 0], sensor_temperatures_c[:, 2])

A same-shaped Boolean mask commonly flattens selected elements into one dimension. If row identity
matters, use a one-dimensional row mask such as `array[row_condition, :]`.

### Views, copies, and mutation

Basic slicing generally returns a **view** sharing the original data buffer. Advanced indexing with
Boolean or integer arrays returns a **copy**. Shared data improves performance, but mutating a view
can change the source unexpectedly.

In [ ]:
view_demo = sensor_temperatures_c.copy()
first_two_times_view = view_demo[:2, :]
selected_times_copy = view_demo[[0, 1], :]

first_two_times_view[0, 0] = -999.0

assert view_demo[0, 0] == -999.0
assert selected_times_copy[0, 0] != -999.0
assert np.shares_memory(view_demo, first_two_times_view)
assert not np.shares_memory(view_demo, selected_times_copy)

Use `.copy()` at an ownership boundary when independent mutation is required. Do not scatter copies
everywhere “for safety”: they consume memory and can conceal an unclear ownership design. The
question is who owns mutation, not merely which method was called.

## 5. Vectorization: express operations on whole arrays

A **universal function** (ufunc) applies an elementwise operation using NumPy's array machinery.
Vectorization often gives clearer mathematical code and avoids a Python-level loop. It does not mean
“never loop”; loops remain appropriate when steps are sequential, irregular, stateful, or dominated
by external work.

In [ ]:
ambient_temperature_c = 22.0

python_excess_c = [
    [temperature - ambient_temperature_c for temperature in row]
    for row in sensor_temperatures_c.tolist()
]
numpy_excess_c = sensor_temperatures_c - ambient_temperature_c

assert np.allclose(numpy_excess_c, np.array(python_excess_c))
assert numpy_excess_c.shape == sensor_temperatures_c.shape

### Broadcasting aligns shapes from the trailing axes

Broadcasting lets compatible shapes participate in elementwise operations without physically
repeating the smaller operand. Compare dimensions from right to left: each pair must be equal, or one
must be `1`, or one dimension is absent.

```text
temperature matrix     (4, 3)
sensor calibration        (3,)
result                 (4, 3)
```

The three calibration values align with the last axis: sensors.

In [ ]:
sensor_calibration_c = np.array([0.10, -0.05, 0.20])
calibrated_temperatures_c = sensor_temperatures_c + sensor_calibration_c

assert calibrated_temperatures_c.shape == (4, 3)
assert np.allclose(
    calibrated_temperatures_c[0],
    np.array([76.50, 72.15, 65.00]),
)

To apply one value per row, add an explicit length-one axis: shape `(4, 1)`. `None` and
`np.newaxis` both create such an axis.

In [ ]:
time_offsets_c = np.array([0.0, 0.1, -0.1, 0.05])
row_offsets_c = time_offsets_c[:, np.newaxis]
time_corrected_c = sensor_temperatures_c + row_offsets_c

assert row_offsets_c.shape == (4, 1)
assert time_corrected_c.shape == (4, 3)

Broadcasting can produce a plausible but scientifically wrong shape. If both “per sensor” and “per
time” vectors have length three in another experiment, NumPy cannot infer which meaning you intended.
Names and explicit shape assertions protect semantics.

In [ ]:
try:
    sensor_temperatures_c + np.array([1.0, 2.0])
except ValueError as error:
    print(type(error).__name__, error)

## 6. Reductions collapse axes

An aggregation without `axis` reduces all elements. With `axis=0`, NumPy removes the time axis and
leaves one result per sensor. With `axis=1`, it removes the sensor axis and leaves one result per time.
The phrase “reduce along axis 0” means axis 0 disappears.

In [ ]:
overall_mean_c = sensor_temperatures_c.mean()
mean_by_sensor_c = sensor_temperatures_c.mean(axis=0)
mean_by_time_c = sensor_temperatures_c.mean(axis=1)

assert np.ndim(overall_mean_c) == 0
assert mean_by_sensor_c.shape == (3,)
assert mean_by_time_c.shape == (4,)
assert np.isclose(mean_by_sensor_c[0], 53.9)

Use `keepdims=True` when the reduced axis must remain as length one for later broadcasting. This can
make standardization easier to reason about.

In [ ]:
column_means_c = sensor_temperatures_c.mean(axis=0, keepdims=True)
column_std_c = sensor_temperatures_c.std(axis=0, ddof=0, keepdims=True)
standardized_temperatures = (
    sensor_temperatures_c - column_means_c
) / column_std_c

assert column_means_c.shape == (1, 3)
assert standardized_temperatures.shape == (4, 3)
assert np.allclose(standardized_temperatures.mean(axis=0), 0.0, atol=1e-12)
assert np.allclose(standardized_temperatures.std(axis=0), 1.0)

The `ddof` choice belongs to the statistical contract. Here `ddof=0` treats these four values as the
complete population being standardized. A sample standard deviation often uses `ddof=1`; neither is
universally correct.

### Floating-point comparisons need a numerical policy

Binary floating-point cannot represent every decimal exactly. Use exact equality for exact discrete
contracts and `np.isclose`/`np.allclose` when rounding error is expected. Tolerances must follow the
algorithm, scale, measurement precision, and consequence—not a desire for green tests.

In [ ]:
decimal_sum = np.array([0.1, 0.2]).sum()

assert decimal_sum != 0.3
assert np.isclose(decimal_sum, 0.3, rtol=1e-12, atol=1e-15)

## 7. Missing numerical values are policy, not decoration

`np.nan` is a floating-point “not a number” value often used to mark missing numerical data. Ordinary
reductions propagate it. `np.nanmean` explicitly ignores it. Ignoring a failed sensor is a scientific
decision, not merely a convenient function call.

In [ ]:
temperatures_with_gap_c = sensor_temperatures_c.copy()
temperatures_with_gap_c[2, 2] = np.nan

assert np.isnan(temperatures_with_gap_c.mean())
assert np.isfinite(np.nanmean(temperatures_with_gap_c))
assert np.isnan(temperatures_with_gap_c).sum() == 1

Ask why the value is missing, whether missingness is informative, and which analyses remain valid.
Preserve a quality flag or provenance field alongside the number. Never replace missing values with
zero unless zero has the intended domain meaning.

## 8. pandas adds labels and heterogeneous columns

A `Series` is a one-dimensional array of values with an `Index`. A `DataFrame` is a two-dimensional
labeled table whose columns may have different dtypes.

```text
DataFrame
                   columns Index
                 ┌───────────────┐
row Index  ─────▶ │ column arrays │
                 │ share row axis│
                 └───────────────┘
```

Think “mapping from column label to aligned `Series`,” not “spreadsheet with magic cells.”

### Build records with explicit provenance

The following cell generates the deterministic teaching observations. A real project would preserve
the immutable raw file, source license, acquisition timestamp, units, and data dictionary. We keep
the generation formula visible so the dataset is reproducible.

In [ ]:
materials = ("copper", "aluminum")
material_factors = {"copper": 1.0, "aluminum": 0.72}
sensor_ids = np.array(["S1", "S2", "S3"])
fixed_deviations_c = np.array(
    [
        0.20,
        -0.10,
        0.05,
        -0.15,
        0.10,
        -0.05,
        0.12,
        -0.08,
        0.03,
        -0.04,
        0.07,
        -0.02,
        -0.18,
        0.11,
        -0.04,
        0.09,
        -0.06,
        0.02,
        -0.10,
        0.08,
        -0.03,
        0.05,
        -0.02,
        0.01,
    ]
)

measurement_records: list[dict[str, object]] = []
deviation_index = 0
for material in materials:
    for time_s in measurement_times_s:
        for sensor_id, distance_cm in zip(
            sensor_ids,
            sensor_distances_cm,
            strict=True,
        ):
            modeled_temperature_c = ambient_temperature_c + (
                60.0
                * material_factors[material]
                * np.exp(-time_s / 75.0)
                * np.exp(-distance_cm / 12.0)
            )
            measurement_records.append(
                {
                    "material": material,
                    "time_s": int(time_s),
                    "sensor_id": str(sensor_id),
                    "distance_cm": float(distance_cm),
                    "temperature_c": round(
                        modeled_temperature_c
                        + fixed_deviations_c[deviation_index],
                        2,
                    ),
                    "quality_flag": "ok",
                }
            )
            deviation_index += 1

measurements = pd.DataFrame.from_records(measurement_records)
measurements.loc[
    (measurements["material"] == "aluminum")
    & (measurements["time_s"] == 60)
    & (measurements["sensor_id"] == "S3"),
    ["temperature_c", "quality_flag"],
] = [np.nan, "sensor_dropout"]

assert measurements.shape == (24, 6)
assert measurements["temperature_c"].isna().sum() == 1
measurements.head(8)

The display is a representation, not the object itself. Start an unfamiliar table with structural
questions: shape, column names, dtypes, missing counts, identifier uniqueness, and representative
rows. `head()` alone cannot establish quality.

In [ ]:
print("shape:", measurements.shape)
print("columns:", measurements.columns.tolist())
print("dtypes:\n", measurements.dtypes)
print("missing:\n", measurements.isna().sum())
print("duplicate complete rows:", measurements.duplicated().sum())

assert measurements.columns.is_unique
assert measurements.duplicated().sum() == 0

### Dtypes are column-level contracts

pandas has NumPy-backed dtypes and extension dtypes, including nullable integers, strings, Booleans,
and categoricals. `object` is very general and often deserves investigation. A column's dtype still
does not encode physical units or all valid values.

In [ ]:
measurements = measurements.astype(
    {
        "material": "category",
        "time_s": "int64",
        "sensor_id": "string",
        "distance_cm": "float64",
        "temperature_c": "float64",
        "quality_flag": "string",
    }
)

assert isinstance(measurements["material"].dtype, pd.CategoricalDtype)
assert isinstance(measurements["sensor_id"].dtype, pd.StringDtype)

Categorical dtype can encode a controlled vocabulary and reduce repeated string storage. Do not mark
a free-text or rapidly evolving field categorical merely because it contains strings.

## 9. `Series`, `Index`, and `DataFrame` selection

Selecting one column with one pair of brackets returns a `Series`; selecting a list of column names
returns a `DataFrame`. The shapes differ.

In [ ]:
temperature_series = measurements["temperature_c"]
temperature_frame = measurements[["temperature_c"]]

assert isinstance(temperature_series, pd.Series)
assert temperature_series.shape == (24,)
assert isinstance(temperature_frame, pd.DataFrame)
assert temperature_frame.shape == (24, 1)

### `.loc` uses labels; `.iloc` uses integer positions

```text
measurements.loc[row_labels_or_mask, column_labels]
measurements.iloc[row_positions, column_positions]
```

The default row labels happen to be integers, but they are labels. After filtering or setting a
scientific identifier as the index, label and position can diverge sharply.

In [ ]:
copper_mask = measurements["material"] == "copper"
copper_late = measurements.loc[
    copper_mask & (measurements["time_s"] >= 60),
    ["time_s", "sensor_id", "temperature_c"],
]
first_two_rows_first_three_columns = measurements.iloc[:2, :3]

assert copper_late.shape == (6, 3)
assert first_two_rows_first_three_columns.shape == (2, 3)
assert (copper_late["time_s"] >= 60).all()

Use parentheses around each comparison joined by `&` or `|`; Python's `and` and `or` ask for one
truth value, while a `Series` contains many. Avoid chained selection such as
`frame[mask]["column"] = value`. Express assignment in one `.loc[...] = ...` operation so the target
and mutation are explicit.

In [ ]:
selection_demo = measurements.copy()
selection_demo.loc[
    selection_demo["quality_flag"] == "sensor_dropout",
    "quality_flag",
] = "missing_sensor_reading"

assert "missing_sensor_reading" in selection_demo["quality_flag"].array
assert "sensor_dropout" in measurements["quality_flag"].array

## 10. pandas aligns by labels, not merely by position

Alignment is one of pandas' most powerful and most surprising behaviors. Arithmetic between `Series`
matches index labels. The result index is normally the union of labels; a label missing on either
side yields a missing result.

In [ ]:
observed_by_sensor = pd.Series(
    {"S1": 50.0, "S2": 47.0, "S3": 43.0},
    name="observed_c",
)
modeled_by_sensor = pd.Series(
    {"S3": 42.5, "S1": 49.5, "S2": 47.5},
    name="modeled_c",
)
residual_by_sensor = observed_by_sensor - modeled_by_sensor

assert residual_by_sensor.index.tolist() == ["S1", "S2", "S3"]
assert residual_by_sensor.loc["S1"] == 0.5
assert residual_by_sensor.loc["S2"] == -0.5
assert residual_by_sensor.loc["S3"] == 0.5

The modeled values were written in a different order, but labels protected identity. Converting both
objects to arrays before subtracting would discard that protection.

In [ ]:
incomplete_model = modeled_by_sensor.drop(index="S2")
incomplete_residual = observed_by_sensor - incomplete_model

assert pd.isna(incomplete_residual.loc["S2"])
assert incomplete_residual.index.equals(observed_by_sensor.index)

Before arithmetic or assignment, ask whether alignment by identity is intended. Assert unique indexes
when uniqueness is required. If positional computation is intended, make the conversion explicit and
verify order immediately beforehand.

In [ ]:
duplicate_labels = pd.Series([1.0, 2.0], index=["S1", "S1"])

assert not duplicate_labels.index.is_unique
assert observed_by_sensor.index.is_unique

## 11. Derived columns preserve the table's analytical context

Column arithmetic aligns on the row index. Use names that carry units. `.assign` can build a new table
without mutating the source; direct `.loc` assignment is appropriate when mutation is intentional.

In [ ]:
analysis_table = measurements.assign(
    excess_temperature_c=lambda frame: (
        frame["temperature_c"] - ambient_temperature_c
    ),
    distance_m=lambda frame: frame["distance_cm"] / 100.0,
)

assert "excess_temperature_c" not in measurements.columns
assert np.isclose(analysis_table.loc[0, "distance_m"], 0.01)
assert analysis_table["excess_temperature_c"].isna().sum() == 1

Mutation and copying differ between NumPy and pandas. pandas 3 uses Copy-on-Write semantics, but this
does not make ownership irrelevant. Prefer transformations whose inputs and outputs are clear, and
use explicit single-step assignment when updating a table.

## 12. Missing data requires a declared treatment

`isna()` detects missing values across `np.nan`, `pd.NA`, and other dtype-specific markers. Common
operations often skip missing values by default, but defaults do not supply a scientific rationale.

In [ ]:
missing_temperature_rows = analysis_table.loc[
    analysis_table["temperature_c"].isna(),
    ["material", "time_s", "sensor_id", "quality_flag"],
]

assert missing_temperature_rows.shape == (1, 4)
assert missing_temperature_rows.iloc[0]["quality_flag"] == "sensor_dropout"

Possible policies include excluding affected observations for a particular summary, modeling
missingness, re-measuring, or imputing from training data inside a leakage-safe pipeline. Each changes
the estimand or uncertainty. Here we retain the row and let grouped means ignore the missing reading,
while reporting the valid count beside every mean.

## 13. Grouping is split–apply–combine

`groupby` partitions rows by key values, applies an operation within each group, and combines labeled
results. The keys define the scientific comparison; the aggregation defines what information is lost.

In [ ]:
material_summary = (
    analysis_table.groupby("material", observed=True)
    .agg(
        mean_temperature_c=("temperature_c", "mean"),
        standard_deviation_c=("temperature_c", "std"),
        valid_measurements=("temperature_c", "count"),
        missing_measurements=("temperature_c", lambda values: values.isna().sum()),
    )
    .reset_index()
)

assert material_summary.shape == (2, 5)
assert material_summary["valid_measurements"].sum() == 23
assert material_summary["missing_measurements"].sum() == 1
material_summary

Notice that `count` excludes missing values while `size` counts rows. We report missingness rather
than allowing the denominators to remain invisible. Standard deviation for groups with too few valid
observations would itself be missing.

### Long and wide forms answer different computational needs

Our table is **long**: each row is one material–time–sensor observation. Pivoting can make a numerical
matrix whose row and column labels preserve meaning. `pivot` requires each requested cell to be
unique; duplicates require an explicit aggregation with `pivot_table`.

In [ ]:
copper_wide = (
    analysis_table.loc[analysis_table["material"] == "copper"]
    .pivot(index="time_s", columns="sensor_id", values="temperature_c")
    .sort_index()
)

assert copper_wide.shape == (4, 3)
assert copper_wide.index.tolist() == [0, 30, 60, 90]
assert copper_wide.columns.tolist() == ["S1", "S2", "S3"]
copper_wide

## 14. Joining tables must state cardinality

Normalized data often stores sensor metadata separately. `merge(..., validate="many_to_one")` makes
the expected relationship executable: many measurements may refer to one sensor record.

In [ ]:
sensor_metadata = pd.DataFrame(
    {
        "sensor_id": pd.Series(["S1", "S2", "S3"], dtype="string"),
        "sensor_model": pd.Series(["T-100", "T-100", "T-200"], dtype="string"),
        "calibration_offset_c": [0.10, -0.05, 0.20],
    }
)

enriched_measurements = analysis_table.merge(
    sensor_metadata,
    on="sensor_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

assert enriched_measurements.shape[0] == analysis_table.shape[0]
assert enriched_measurements["_merge"].eq("both").all()
assert enriched_measurements["sensor_model"].notna().all()

Without cardinality validation, duplicated sensor metadata could multiply rows and silently change
statistics. Always check unmatched keys and row-count expectations. A successful join establishes
structural compatibility, not that the metadata is scientifically correct.

## 15. Crossing the pandas–NumPy boundary

Many numerical and machine-learning APIs accept arrays. Select columns in a deliberate order, check
missingness, and then call `.to_numpy()`. The resulting array does not retain column or row labels.

In [ ]:
feature_names = ["time_s", "distance_cm"]
feature_frame = analysis_table.loc[:, feature_names]
feature_matrix = feature_frame.to_numpy(dtype=np.float64, copy=True)

assert feature_frame.columns.tolist() == feature_names
assert not feature_frame.isna().any().any()
assert feature_matrix.shape == (24, 2)
assert feature_matrix.dtype == np.float64

Store `feature_names` with a trained artifact and validate them at inference. Shape `(n, 2)` cannot
tell a model whether column 0 is time or distance. A DataFrame is not automatically safe either:
explicit reindexing or schema validation is needed when column order forms part of a public data contract.

In [ ]:
reversed_feature_frame = feature_frame.loc[:, list(reversed(feature_names))]

assert reversed_feature_frame.shape == feature_frame.shape
assert not np.array_equal(
    reversed_feature_frame.to_numpy(),
    feature_matrix,
)
assert reversed_feature_frame.columns.tolist() != feature_names

## 16. A reusable numerical interface should validate shape and values

The next function standardizes each feature column. Its annotation says “array,” while its docstring
states semantics that the type cannot: two dimensions, observations by features, finite values, and
nonconstant columns. Runtime checks enforce those boundary properties.

In [ ]:
def standardize_feature_matrix(values: np.ndarray) -> np.ndarray:
    """Standardize each feature column to population mean zero and variance one.

    Parameters
    ----------
    values : numpy.ndarray
        Two-dimensional floating-point array with observations on axis 0 and
        features on axis 1. Every value must be finite, and every feature must
        have positive population standard deviation.

    Returns
    -------
    numpy.ndarray
        New floating-point array with the same shape as ``values``.

    Raises
    ------
    TypeError
        If ``values`` is not a NumPy array.
    ValueError
        If the array is not two-dimensional, contains non-finite values, has
        no observations or features, or contains a constant feature.

    Notes
    -----
    In a machine-learning workflow, means and scales must be fitted on training
    data only and reused unchanged on validation, test, and production data.
    """

    if not isinstance(values, np.ndarray):
        raise TypeError("values must be a numpy.ndarray")
    if values.ndim != 2:
        raise ValueError(
            "values must be a two-dimensional array with shape "
            "(observations, features)"
        )
    if 0 in values.shape:
        raise ValueError("values must contain observations and features")
    if not np.issubdtype(values.dtype, np.number):
        raise TypeError("values must have a numerical dtype")

    floating_values = values.astype(np.float64, copy=True)
    if not np.isfinite(floating_values).all():
        raise ValueError("values must contain only finite numbers")

    feature_means = floating_values.mean(axis=0, keepdims=True)
    feature_scales = floating_values.std(axis=0, ddof=0, keepdims=True)
    if np.any(feature_scales == 0.0):
        raise ValueError("every feature must have positive standard deviation")

    return (floating_values - feature_means) / feature_scales


standardized_features = standardize_feature_matrix(feature_matrix)

assert standardized_features.shape == feature_matrix.shape
assert not np.shares_memory(standardized_features, feature_matrix)
assert np.allclose(standardized_features.mean(axis=0), 0.0, atol=1e-12)
assert np.allclose(standardized_features.std(axis=0), 1.0)

This pedagogical function demonstrates the mathematics. In a production ML pipeline, use a tested
transformer that stores fitted means and scales, and fit it only on training data. Recomputing the
statistics on evaluation data leaks information and changes the coordinate system.

In [ ]:
with pytest.raises(ValueError, match="two-dimensional"):
    standardize_feature_matrix(np.array([1.0, 2.0]))

with pytest.raises(ValueError, match="finite"):
    standardize_feature_matrix(np.array([[1.0, np.nan], [2.0, 3.0]]))

with pytest.raises(ValueError, match="positive standard deviation"):
    standardize_feature_matrix(np.array([[1.0, 4.0], [1.0, 5.0]]))

## 17. Test tables at several layers

Useful tabular checks include:

- **schema:** required names, dtypes, and key uniqueness;
- **structural:** row counts, shapes, join cardinality, and label order;
- **semantic:** units, permitted categories, physical ranges, and time ordering;
- **missingness:** counts and reasons by field or subgroup;
- **numerical:** finite results, tolerances, invariants, and baselines; and
- **experimental:** transformations fitted only on allowed data and entities kept across boundaries.

No library can infer all these contracts from the data alone.

In [ ]:
required_columns = {
    "material",
    "time_s",
    "sensor_id",
    "distance_cm",
    "temperature_c",
    "quality_flag",
}
observation_key = ["material", "time_s", "sensor_id"]

assert required_columns <= set(measurements.columns)
assert not measurements.duplicated(subset=observation_key).any()
assert measurements["distance_cm"].gt(0).all()
assert measurements["time_s"].ge(0).all()
assert measurements["temperature_c"].dropna().between(20.0, 100.0).all()

The temperature range above is a teaching assumption, not a universal physical law. In real work,
record who owns each rule, its unit and justification, and what response follows a violation.

## 18. Debugging array and table failures

Use a repeatable sequence:

1. reduce the failure to the smallest representative input;
2. print `type`, shape, dtype, and labels—not the entire large object;
3. state what every axis, row, column, and unit should mean;
4. inspect missing, duplicated, non-finite, and out-of-range values;
5. trace whether selection uses labels or positions;
6. write operand shapes right-aligned before debugging broadcasting;
7. check view/copy ownership before and after mutation;
8. verify group keys, aggregation denominators, and join cardinality;
9. cross-check a small result by hand; and
10. turn the discovered failure into a focused regression test.

### Common failure modes

| Symptom | Likely cause | First check |
| --- | --- | --- |
| unexpected string array | dtype promotion during construction | `array.dtype` and raw values |
| broadcasting error | incompatible trailing dimensions | both operand shapes |
| plausible wrong matrix | semantically wrong broadcast axis | axis meaning and explicit singleton axes |
| source changes after slicing | slice is a shared view | `np.shares_memory` and ownership |
| many `NaN` after arithmetic | pandas aligned different labels | both indexes and uniqueness |
| one-column result changes type | `Series` versus `DataFrame` selection | `type` and shape |
| rows multiply after merge | unexpected many-to-many relationship | key duplicates and `validate=` |
| mean uses fewer rows | missing values skipped | valid count beside aggregate |
| model accepts wrong features | labels lost at array boundary | stored feature names and order |
| notebook works only after rerun | hidden state or mutation | restart and run all |

## Guided practice: predict shape, dtype, and ownership

Without running code, analyze:

```python
experiment = np.arange(24, dtype=np.float64).reshape(2, 4, 3)
selection = experiment[:, 1:3, 0]
selection[0, 0] = -1
```

1. Give a domain interpretation for all three axes.
2. Predict `experiment.shape` and `selection.shape`.
3. Decide whether `selection` shares memory with `experiment`.
4. Identify which element changes.

**Success criterion:** justify each answer from integer-versus-slice indexing and view rules, then
verify with assertions.

In [ ]:
practice_experiment = np.arange(24, dtype=np.float64).reshape(2, 4, 3)
practice_selection = practice_experiment[:, 1:3, 0]

assert practice_experiment.shape == (2, 4, 3)
assert practice_selection.shape == (2, 2)
assert np.shares_memory(practice_experiment, practice_selection)

practice_selection[0, 0] = -1.0
assert practice_experiment[0, 1, 0] == -1.0

## Guided practice: diagnose label alignment

Predict the result index and values before running:

```python
left = pd.Series([10.0, 20.0], index=["sample-b", "sample-a"])
right = pd.Series([1.0, 2.0], index=["sample-a", "sample-c"])
left - right
```

Then design two alternatives: one that requires identical identifiers and one that intentionally
keeps the union.

**Success criterion:** explain why equal lengths would not make these observations aligned.

In [ ]:
left_measurements = pd.Series(
    [10.0, 20.0],
    index=["sample-b", "sample-a"],
)
right_measurements = pd.Series(
    [1.0, 2.0],
    index=["sample-a", "sample-c"],
)
aligned_difference = left_measurements - right_measurements

assert aligned_difference.index.tolist() == [
    "sample-a",
    "sample-b",
    "sample-c",
]
assert aligned_difference.loc["sample-a"] == 19.0
assert aligned_difference.loc[["sample-b", "sample-c"]].isna().all()

## Independent practice: validate a measurement table

Write `validate_measurements(frame: pd.DataFrame) -> None`. Require the six source columns used here,
a unique material–time–sensor key, nonnegative time, positive distance, known materials, and a quality
flag whenever temperature is missing. Raise specific exceptions with actionable messages.

Test a valid table plus missing-column, duplicate-key, invalid-range, unknown-category, and unexplained
missing-value cases.

**Success criterion:** the function does not mutate its input, the docstring states units and key
semantics, and each invalid partition has a focused test.

In [ ]:
# Start the independent exercise by preserving an untouched example.
practice_measurements = measurements.copy()
practice_fingerprint = pd.util.hash_pandas_object(
    practice_measurements,
    index=True,
).copy()

# Implement and call validate_measurements(practice_measurements) here.

assert practice_fingerprint.equals(
    pd.util.hash_pandas_object(practice_measurements, index=True)
)

## Independent practice: summarize without hiding missingness

Create a function that groups by material and time and returns mean temperature, standard deviation,
valid count, missing count, and total row count. Specify ordering and output column names.

**Success criterion:** counts reconcile for every group, input rows are unchanged, the one sensor
dropout remains visible, and a hand calculation verifies one small group.

## Extension: reproducible random arrays

Use a local `Generator` from `np.random.default_rng(seed)` for simulated data. A seed makes a specific
pseudorandom stream reproducible; it does not prove robustness or justify a probability model.

In [ ]:
simulation_rng = np.random.default_rng(seed=577)
simulated_sensor_noise_c = simulation_rng.normal(
    loc=0.0,
    scale=0.1,
    size=sensor_temperatures_c.shape,
)

assert simulated_sensor_noise_c.shape == sensor_temperatures_c.shape
assert np.isfinite(simulated_sensor_noise_c).all()

## Extension: design the next package boundary

Decide whether `standardize_feature_matrix` belongs in the notebook or `src/rice_dsm/`. Consider:

- reuse across notebooks;
- need to retain fitted means, scales, and feature names;
- leakage-safe `fit` versus `transform` behavior;
- pandas versus NumPy input contracts;
- serialization and versioning;
- tests for constant columns, non-finite values, and schema reorder; and
- whether an established library already supplies the desired interface.

**Success criterion:** propose a public API, its non-goals, state ownership, and three highest-risk
tests. More than one design is defensible.

## Retrieval practice

Answer without executing code:

1. What do `ndim`, `shape`, `size`, and `dtype` each describe?
2. Why does integer indexing remove an axis while slicing preserves it?
3. Compare basic slicing and advanced indexing with respect to shape and memory.
4. Apply the broadcasting rules to shapes `(8, 1, 3)` and `(4, 3)`.
5. What does `mean(axis=0)` remove from a `(time, sensor)` array?
6. Why can two equal-length pandas `Series` produce missing values when added?
7. When should you choose `.loc` over `.iloc`?
8. Why should an aggregate report valid and missing counts?
9. What information is lost in `DataFrame.to_numpy()`?
10. Why must standardization parameters be learned only from training data?

## Takeaway

```text
NumPy ndarray = homogeneous values + dtype + shape + memory layout
pandas Series = one value array + one label index
pandas DataFrame = aligned labeled columns, potentially with different dtypes
```

Choose NumPy for rectangular numerical computation and pandas for labeled tabular reasoning. At every
boundary, verify shape, axis meaning, dtype, units, labels, uniqueness, missingness, and ownership.
Vectorization and alignment are powerful precisely because they encode rules—rules you must be able
to state and test.

The next notebook will deepen NumPy indexing, reshaping, broadcasting, and vectorized computation.

## Further reading

- [NumPy: absolute basics for beginners](https://numpy.org/doc/stable/user/absolute_beginners.html)
- [NumPy broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html)
- [NumPy copies and views](https://numpy.org/doc/stable/user/basics.copies.html)
- [pandas: introduction to data structures](https://pandas.pydata.org/docs/user_guide/dsintro.html)
- [pandas indexing and selection](https://pandas.pydata.org/docs/user_guide/indexing.html)
- [pandas missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- [pandas Copy-on-Write](https://pandas.pydata.org/docs/user_guide/copy_on_write.html)
- [Python `pathlib`](https://docs.python.org/3/library/pathlib.html)